In [1]:
from pathlib import Path
import pandas as pd

# Generate merged summary results CSV

In [2]:
# Model names to look for (folders are expected as results/grid_search_{model}_AUC-APS/)
models = ['ProtT5', 'ProstT5', 'ESM2', 'esmc_600m', 'esmc_300m']

out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

dfs = []
for model in models:
    d = Path(f'results/grid_search_{model}_AUC-APS')
    f = d / 'summary_results.csv'
    if f.exists():
        df = pd.read_csv(f)
        df['model'] = model
        dfs.append(df)
    else:
        print(f'Missing: {f}')

if len(dfs) == 0:
    print('No summary_results.csv files found.')
else:
    all_df = pd.concat(dfs, ignore_index=True, sort=False)
    out_file = out_dir / 'summary_results.csv'
    all_df.to_csv(out_file, index=False)
    print(f'Saved concatenated CSV to: {out_file} (shape: {all_df.shape})')

Saved concatenated CSV to: results/summary_results.csv (shape: (1250, 18))


# Results and analysis for grid search experiments

## add useful columns

In [3]:
# Read csv
out_dir = Path('results')
summary_csv = out_dir / 'summary_results.csv'
df = pd.read_csv(summary_csv)
print(f'Read summary CSV: {summary_csv} (shape: {df.shape})')

Read summary CSV: results/summary_results.csv (shape: (1250, 18))


In [4]:
# add n_trial column from trial column (where trial is like 'trial0_filt400_ker7_resnet4_win40_lr6e-03_drop0.40_02-03-2026_16-54-06')
df['n_trial'] = df['trial'].str.extract(r'trial(\d+)').astype(int)

In [5]:
# add a column is_best_trial which is True 
best_trials = {
    'ProtT5': {181, 237, 75},        # ejemplo: marcar trial 0 y 2 como mejores
    'ProstT5': {101, 76},
    'ESM2': {221, 152, 89},
    'esmc_600m': {63},
    'esmc_300m': {166},
}
df['is_best_trial'] = df.apply(lambda r: r['n_trial'] in best_trials.get(r['model'], set()), axis=1)

# rename
df.rename(columns={'caid3_3_disorder_pdb_aps': 'pdb_aps'}, inplace=True)
df.rename(columns={'caid3_3_disorder_pdb_auc': 'pdb_auc'}, inplace=True)
df.rename(columns={'caid3_3_disorder_nox_aps': 'nox_aps'}, inplace=True)
df.rename(columns={'caid3_3_disorder_nox_auc': 'nox_auc'}, inplace=True)

In [6]:
df

,trial,filters,kernel_size,lr,n_resnet,p_dropout,win_len,dev_auc,dev_aps,pdb_auc,pdb_aps,nox_auc,nox_aps,trial_number,optuna_dev_auc,optuna_dev_aps,state,model,n_trial,is_best_trial
0,trial0_filt400_ker7_resnet4_win40_lr6e-03_drop...,400,7,0.005510,4,0.4,40,0.862,0.797,0.915,0.893,0.817,0.597,0,0.862,0.797,COMPLETE,ProtT5,0,False
1,trial100_filt150_ker9_resnet4_win100_lr2e-06_d...,150,9,0.000002,4,0.0,100,0.886,0.830,0.919,0.895,0.836,0.580,100,0.886,0.830,COMPLETE,ProtT5,100,False
2,trial101_filt400_ker3_resnet4_win80_lr9e-06_dr...,400,3,0.000009,4,0.3,80,0.889,0.834,0.926,0.902,0.847,0.621,101,0.889,0.834,COMPLETE,ProtT5,101,False
3,trial102_filt50_ker11_resnet4_win35_lr1e-04_dr...,50,11,0.000125,4,0.0,35,0.909,0.848,0.941,0.920,0.842,0.592,102,0.909,0.848,COMPLETE,ProtT5,102,False
4,trial103_filt300_ker11_resnet1_win30_lr6e-05_d...,300,11,0.000059,1,0.1,30,0.917,0.863,0.945,0.919,0.850,0.615,103,0.917,0.863,COMPLETE,ProtT5,103,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,trial96_filt200_ker7_resnet2_win45_lr9e-03_dro...,200,7,0.008505,2,0.4,45,0.900,0.835,0.946,0.919,0.860,0.648,96,0.900,0.835,COMPLETE,esmc_300m,96,False
1246,trial97_filt100_ker9_resnet4_win55_lr1e-06_dro...,100,9,0.000001,4,0.3,55,0.899,0.846,0.941,0.922,0.835,0.639,97,0.899,0.846,COMPLETE,esmc_300m,97,False
1247,trial98_filt100_ker7_resnet4_win15_lr6e-05_dro...,100,7,0.000059,4,0.3,15,0.907,0.840,0.947,0.922,0.848,0.658,98,0.907,0.840,COMPLETE,esmc_300m,98,False
1248,trial99_filt50_ker11_resnet1_win75_lr8e-05_dro...,50,11,0.000077,1,0.1,75,0.894,0.839,0.931,0.899,0.843,0.631,99,0.894,0.839,COMPLETE,esmc_300m,99,False


## Analizamos los resultados

In [ ]:
# muestro solo algunas columnas 
cols = ['model', 'n_trial', 'is_best_trial', 'pdb_auc', 'pdb_aps','dev_auc', 'dev_aps']
hyperparams = ['filters', 'kernel_size', 'n_resnet', 'win_len', 'lr', 'p_dropout']

df_best = df[cols]

# filter only best trials
df_best = df_best[df_best['is_best_trial'] == True]
df_best

,model,n_trial,is_best_trial,pdb_auc,pdb_aps,dev_auc,dev_aps
90,ProtT5,181,True,0.946,0.922,0.921,0.861
152,ProtT5,237,True,0.947,0.921,0.919,0.866
222,ProtT5,75,True,0.943,0.914,0.917,0.868
252,ProstT5,101,True,0.950,0.925,0.914,0.858
473,ProstT5,76,True,0.948,0.924,0.912,0.862
558,ESM2,152,True,0.950,0.924,0.918,0.861
635,ESM2,221,True,0.956,0.927,0.921,0.860
737,ESM2,89,True,0.950,0.924,0.916,0.862
959,esmc_600m,63,True,0.952,0.930,0.920,0.866
1073,esmc_300m,166,True,0.943,0.921,0.917,0.864


In [11]:
df_sel = df[cols + hyperparams]
df_sel

,model,n_trial,is_best_trial,pdb_auc,pdb_aps,dev_auc,dev_aps,filters,kernel_size,n_resnet,win_len,lr,p_dropout
0,ProtT5,0,False,0.915,0.893,0.862,0.797,400,7,4,40,0.005510,0.4
1,ProtT5,100,False,0.919,0.895,0.886,0.830,150,9,4,100,0.000002,0.0
2,ProtT5,101,False,0.926,0.902,0.889,0.834,400,3,4,80,0.000009,0.3
3,ProtT5,102,False,0.941,0.920,0.909,0.848,50,11,4,35,0.000125,0.0
4,ProtT5,103,False,0.945,0.919,0.917,0.863,300,11,1,30,0.000059,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,esmc_300m,96,False,0.946,0.919,0.900,0.835,200,7,2,45,0.008505,0.4
1246,esmc_300m,97,False,0.941,0.922,0.899,0.846,100,9,4,55,0.000001,0.3
1247,esmc_300m,98,False,0.947,0.922,0.907,0.840,100,7,4,15,0.000059,0.3
1248,esmc_300m,99,False,0.931,0.899,0.894,0.839,50,11,1,75,0.000077,0.1
